# Student Performance — Exploratory Analysis & Predictive Modeling

This notebook explores student assessment scores and trains baseline regression models to estimate **Math Score** from the available demographic, preparation, reading, and writing fields. The model is for educational exploration, not for making decisions about individual students.

## 1. Setup and load data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

sns.set_theme(style="whitegrid")
DATA_PATH = Path("data/StudentsPerformance.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Place StudentsPerformance.csv in the project's data/ folder.")

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
score_cols = ["Math Score", "Reading Score", "Writing Score"]
required = {"Gender", "Race/Ethnicity", "Parental Level Of Education", "Lunch", "Test Preparation Course", *score_cols}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing expected columns: {sorted(missing)}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())

## 2. Data quality and summary

In [ ]:
display(df.info())
display(df.describe(include="all").T)
quality = pd.DataFrame({"missing": df.isna().sum(), "unique_values": df.nunique()})
display(quality)
print("Duplicate rows:", int(df.duplicated().sum()))

## 3. Score distributions and relationships

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, score_cols):
    sns.histplot(data=df, x=col, bins=20, kde=True, ax=ax)
    ax.set_title(f"{col} distribution")
plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 4))
sns.heatmap(df[score_cols].corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Correlation between assessment scores")
plt.tight_layout(); plt.show()

## 4. Compare scores across groups

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x="Test Preparation Course", y="Math Score", ax=axes[0])
axes[0].set_title("Math score by test preparation")
sns.boxplot(data=df, x="Lunch", y="Math Score", ax=axes[1])
axes[1].set_title("Math score by lunch type")
plt.tight_layout(); plt.show()

group_summary = df.groupby("Test Preparation Course", observed=True)[score_cols].agg(["mean", "median", "count"]).round(2)
display(group_summary)

## 5. Prepare data and compare regression models

This baseline uses an 80/20 train-test split with a fixed random seed. Categorical fields are one-hot encoded inside a pipeline to keep preprocessing within the training workflow.

In [ ]:
target = "Math Score"
X = df.drop(columns=[target])
y = df[target]
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()

preprocess = ColumnTransformer([
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), num_cols),
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=6, min_samples_leaf=5, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1),
}
results, fitted = [], {}
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = pipe
    results.append({"Model": name,
                    "MAE": mean_absolute_error(y_test, pred),
                    "RMSE": mean_squared_error(y_test, pred) ** 0.5,
                    "R²": r2_score(y_test, pred)})
metrics = pd.DataFrame(results).sort_values("MAE")
display(metrics.round(3))

## 6. Actual vs. predicted scores

In [ ]:
model_name = metrics.iloc[0]["Model"]
best_model = fitted[model_name]
pred = best_model.predict(X_test)
plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_test, y=pred, alpha=.65)
plt.plot([0, 100], [0, 100], linestyle="--")
plt.xlim(0, 100); plt.ylim(0, 100)
plt.xlabel("Actual Math Score"); plt.ylabel("Predicted Math Score")
plt.title(f"Actual vs. predicted — {model_name}")
plt.tight_layout(); plt.show()
print("Displayed model:", model_name)

## 7. Interpretation and limitations

- The dataset contains 1,000 student records and eight columns.
- Associations between group attributes and scores are descriptive; they do not establish that a demographic attribute causes a score difference.
- Reading and writing scores are included as predictors. In a real use case, they must be available at the time the math score is to be estimated; otherwise, this setup would not match the intended prediction scenario.
- The dataset is small, and a single train-test split can produce variable estimates. Validate on additional data before making broader claims.
- Do not use this model to determine admissions, placement, discipline, or access to educational opportunities.